# 📐 鏡頭可視範圍 (FOV) 計算原理
---
本工具基於三角幾何模型，協助開發者精確計算自走車支架高度與實際拍攝範圍。  

### 參數定義 (Parameters)
| 符號 | 中文名稱 | 說明 |
| :--- | :--- | :--- |
| $H_{total}$ | **支架高度** | 鏡頭到地面的垂直距離 |
| $H_{target}$ | **目標高度** | 壟位 + 草莓植株的高度 (預設 0.5m) |
| $H_{work}$ | **工作高度** | 實際光學工作距離 ($H_{total} - H_{target}$) |
| $\alpha$ | **俯視角度** | 鏡頭與水平線的夾角 ($90^\circ$ 為正俯拍) |
| $\theta_{h}, \theta_{v}$ | **水平/垂直視角** | 鏡頭在水平與垂直方向的可視角度 (Degrees) |
| $\theta_{raw}$ | **原始視角** | 鏡頭硬體標稱的視角 (標註為 `raw` 非自然對數 $e$) |
| $\theta_{eff}$ | **有效視角** | 考慮縮放倍率後的實際視角 (標註為 `eff`) |
| $Z$ | **縮放倍率** | 數位變焦倍數 (Zoom) |

---
### 🔢 核心公式 (Formulas)

#### 1. 有效視角修正 (Effective FOV)
考慮數位縮放對可視角度的影響：
$$\tan\left(\frac{\theta_{eff}}{2}\right) = \frac{\tan\left(\frac{\theta_{raw}}{2}\right)}{Z}$$

#### 2. 地面涵蓋寬度 ($W$) 與長度 ($L$) 計算
基於幾何投影，計算目標物高度處的涵蓋範圍：
$$W = \frac{2 \cdot H_{work} \cdot \tan\left(\frac{\theta_{h\_eff}}{2}\right)}{\sin(\alpha)}$$
$$L = \frac{2 \cdot H_{work} \cdot \tan\left(\frac{\theta_{v\_eff}}{2}\right)}{\sin(\alpha)}$$

#### 3. 建議支架高度 ($H_{total}$)
若已知所需涵蓋寬度 $W$，建議之架設高度為：
$$H_{total} = \frac{W \cdot \sin(\alpha)}{2 \cdot \tan\left(\frac{\theta_{h\_eff}}{2}\right)} + H_{target}$$

---

In [4]:
import math

# ==========================================
# 📐 鏡頭範圍與高度計算器 (專業彙報版)
# ==========================================

PRESET_OPTIONS = [
    {"name": "Logitech C920 (16:9)", "h_fov": 70.4, "v_fov": 43.3},
    {"name": "DJI Mini 4 Pro (16:9)", "h_fov": 75.0, "v_fov": 46.5},
    {"name": "DJI Mini 4 Pro (4:3)", "h_fov": 71.2, "v_fov": 56.2},
    {"name": "iPhone 15 Main (16:9)", "h_fov": 73.7, "v_fov": 45.4},
    {"name": "RPi Camera V3 (16:9)", "h_fov": 66.0, "v_fov": 41.0},
    {"name": "Custom", "h_fov": 84.0, "v_fov": 60.0}
]

print("--- 🛠️ 功能模式選擇 ---")
print("[1] 已知『所需區域大小』 -> 計算『支架高度』")
print("[2] 已知『支架高度』     -> 計算『實際畫面涵蓋範圍』")

try:
    calc_mode = int(input("請選擇功能模式 (1-2, 預設 1): ") or 1)
    mode_name = "已知區域求高度" if calc_mode == 1 else "已知高度求範圍"
    
    print("\n--- 📱 拍攝方向選擇 ---")
    print("[1] 橫向拍攝 (Landscape)")
    print("[2] 直向拍攝 (Portrait)")
    orient = int(input("請選擇拍攝方向 (1-2, 預設 1): ") or 1)
    orient_name = "橫向拍攝 (Landscape)" if orient == 1 else "直向拍攝 (Portrait)"

    print("\n--- 🎥 鏡頭參數設定 ---")
    for i, opt in enumerate(PRESET_OPTIONS):
        print(f"[{i+1}] {opt['name']}")
    
    cam_choice = int(input(f"請輸入鏡頭編號 (1-{len(PRESET_OPTIONS)}, 預設 3): ") or 3)
    selected = PRESET_OPTIONS[cam_choice-1]
    
    if orient == 1:
        use_h_fov, use_v_fov = selected['h_fov'], selected['v_fov']
    else:
        # 直拍：交換水平與垂直視角
        use_h_fov, use_v_fov = selected['v_fov'], selected['h_fov']

    print(f"\n--- 📐 拍攝環境設定 ({orient_name}) ---")
    angle = float(input("請輸入拍攝俯視角度 (度, 90為正俯拍, 預設 90): ") or 90)
    obj_h = float(input("請輸入『目標物高度』 (m, 預設 0.5): ") or 0.5)
    zoom  = float(input("請輸入縮放倍率 (預設 1.0): ") or 1.0)

    eff_tan_h = math.tan(math.radians(use_h_fov / 2)) / zoom
    eff_tan_v = math.tan(math.radians(use_v_fov / 2)) / zoom

    if calc_mode == 1:
        target_w = float(input("請輸入希望涵蓋的『橫向寬度』 (m, 左右寬度, 預設 23.0): ") or 23.0)
        target_l = float(input("請輸入希望涵蓋的『縱向長度』 (m, 上下長度, 若無請填0, 預設 18.0): ") or 18.0)
        
        req_h_w = (target_w * math.sin(math.radians(angle)) / (2 * eff_tan_h)) + obj_h
        if target_l > 0:
            req_h_l = (target_l * math.sin(math.radians(angle)) / (2 * eff_tan_v)) + obj_h
        else:
            req_h_l = 0
            
        req_h = max(req_h_w, req_h_l)
        slant_range = (req_h - obj_h) / math.sin(math.radians(angle))
        bottleneck = "橫向寬度限制" if req_h_w >= req_h_l else "縱向長度限制"
        
        print("\n==========================================")
        print("📋 輸入參數彙總：")
        print(f"   - 計算模式: {mode_name}")
        print(f"   - 拍攝方向: {orient_name}")
        print(f"   - 使用鏡頭: {selected['name']}")
        print(f"   - 俯視角度: {angle}°")
        print(f"   - 目標橫向寬度: {target_w} m")
        if target_l > 0:
            print(f"   - 目標縱向長度: {target_l} m")
        print("------------------------------------------")
        print(f"✅ 計算完畢 (高度受『{bottleneck}』影響)：")
        print(f"   - 🚀 建議『支架垂直高度』 (H): {req_h:.2f} 公尺 (約 {req_h*100:.1f} 公分)")
        print(f"   - 📏 鏡頭到目標中心直線距離: {slant_range:.2f} 公尺")
        
        # 顯示在這個高度下，實際拍出來的畫面大小
        actual_w = (req_h - obj_h) * 2 * eff_tan_h / math.sin(math.radians(angle))
        actual_l = (req_h - obj_h) * 2 * eff_tan_v / math.sin(math.radians(angle))
        print("------------------------------------------")
        print(f"🖼️ 此高度下實際畫面涵蓋範圍：")
        print(f"   - 實際橫向涵蓋寬度: {actual_w:.2f} 公尺")
        print(f"   - 實際縱向涵蓋長度: {actual_l:.2f} 公尺")
        print("==========================================")
        
    else:
        input_h = float(input("請輸入目前的『支架垂直高度』 (m, 預設 17.5): ") or 17.5)
        work_h = input_h - obj_h
        
        print("\n==========================================")
        print("📋 輸入參數彙總：")
        print(f"   - 計算模式: {mode_name}")
        print(f"   - 拍攝方向: {orient_name}")
        print(f"   - 使用鏡頭: {selected['name']}")
        print(f"   - 俯視角度: {angle}°")
        print(f"   - 支架高度: {input_h} m")
        print("------------------------------------------")
        
        if work_h <= 0:
            print("❌ 錯誤：支架高度必須大於目標物高度！")
        else:
            fov_w = (2 * work_h * eff_tan_h) / math.sin(math.radians(angle))
            fov_h = (2 * work_h * eff_tan_v) / math.sin(math.radians(angle))
            
            print(f"✅ 計算完畢：")
            print(f"   - 🖼️ 實際畫面涵蓋『寬度』: {fov_w:.2f} 公尺 (約 {fov_w*100:.1f} 公分)")
            print(f"   - 🖼️ 實際畫面涵蓋『長度』: {fov_h:.2f} 公尺 (約 {fov_h*100:.1f} 公分)")
            print(f"   - 💡 提示：若為傾斜拍攝，實際長度範圍可能會因透視形變而略有增減。")
        print("==========================================")

except Exception as e:
    print(f"\n❌ 發生錯誤: {e}\n")


--- 🛠️ 功能模式選擇 ---
[1] 已知『所需區域大小』 -> 計算『支架高度』
[2] 已知『支架高度』     -> 計算『實際畫面涵蓋範圍』

--- 📱 拍攝方向選擇 ---
[1] 橫向拍攝 (Landscape)
[2] 直向拍攝 (Portrait)

--- 🎥 鏡頭參數設定 ---
[1] Logitech C920 (16:9)
[2] DJI Mini 4 Pro (16:9)
[3] DJI Mini 4 Pro (4:3)
[4] iPhone 15 Main (16:9)
[5] RPi Camera V3 (16:9)
[6] Custom

--- 📐 拍攝環境設定 (橫向拍攝 (Landscape)) ---

📋 輸入參數彙總：
   - 計算模式: 已知區域求高度
   - 拍攝方向: 橫向拍攝 (Landscape)
   - 使用鏡頭: DJI Mini 4 Pro (4:3)
   - 俯視角度: 90.0°
   - 目標橫向寬度: 23.0 m
   - 目標縱向長度: 18.0 m
------------------------------------------
✅ 計算完畢 (高度受『縱向長度限制』影響)：
   - 🚀 建議『支架垂直高度』 (H): 17.36 公尺 (約 1735.6 公分)
   - 📏 鏡頭到目標中心直線距離: 16.86 公尺
------------------------------------------
🖼️ 此高度下實際畫面涵蓋範圍：
   - 實際橫向涵蓋寬度: 24.13 公尺
   - 實際縱向涵蓋長度: 18.00 公尺


In [ ]:
import os, glob, cv2, datetime
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont

# ==========================================
# 🖼️ 圖片推論 (修正重複讀取問題)
# ==========================================
TEST_PATH = 'roboflow_側拍/色階排序'
CONF_LEVEL = 0.5
SAVE_DIR = '診斷結果'
IOU_THRESHOLD = 0.45

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

def draw_chinese_text(img, text, position, color=(0, 0, 255), size=30):
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font_paths = ["C:\\Windows\\Fonts\\msjh.ttc", "C:\\Windows\\Fonts\\msjhbd.ttc", "arial.ttf"]
    font = None
    for path in font_paths:
        if os.path.exists(path):
            font = ImageFont.truetype(path, size)
            break
    if font is None: font = ImageFont.load_default()
    draw.text(position, text, font=font, fill=color[::-1]) 
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

def run_image_inference(img_path, model):
    if not os.path.exists(SAVE_DIR): os.makedirs(SAVE_DIR)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    results = model.predict(img_path, conf=CONF_LEVEL, imgsz=640, verbose=False)
    pred_boxes = results[0].boxes.xyxy.cpu().numpy()
    pred_cls = results[0].boxes.cls.cpu().numpy().astype(int)
    pred_img_bgr = results[0].plot() 
    
    txt_path = os.path.splitext(img_path)[0] + ".txt"
    gt_data = []
    if os.path.exists(txt_path):
        h, w = cv2.imread(img_path).shape[:2]
        with open(txt_path, 'r') as f:
            for line in f.readlines():
                parts = line.split()
                if len(parts) >= 5:
                    c, x, y, nw, nh = map(float, parts[:5])
                    gt_data.append((int(c), (x-nw/2)*w, (y-nh/2)*h, (x+nw/2)*w, (y+nh/2)*h))

    print(f"\n🔍 處理圖片: {os.path.basename(img_path)}")
    if gt_data:
        matched_gt = [False] * len(gt_data)
        matched_pred = [False] * len(pred_boxes)
        cls_error_indices = []
        for i, gt in enumerate(gt_data):
            for j, pred in enumerate(pred_boxes):
                iou = calculate_iou(gt[1:], pred)
                if iou > IOU_THRESHOLD:
                    matched_gt[i] = True
                    matched_pred[j] = True
                    if gt[0] != pred_cls[j]:
                        cls_error_indices.append((i, j))
        
        missed_indices = [i for i, m in enumerate(matched_gt) if not m]
        extra_indices = [j for j, m in enumerate(matched_pred) if not m]
        
        print(f"📊 報告: GT:{len(gt_data)} | Pred:{len(pred_boxes)} | 🔴漏抓:{len(missed_indices)} | 🟡誤判:{len(cls_error_indices)} | 🟠多抓:{len(extra_indices)}")
            
        orig_img = cv2.imread(img_path)
        gt_img = orig_img.copy()
        for gt in gt_data: cv2.rectangle(gt_img, (int(gt[1]), int(gt[2])), (int(gt[3]), int(gt[4])), (0, 255, 0), 3)
        
        err_img = orig_img.copy()
        for idx in missed_indices:
            box = gt_data[idx][1:]
            cv2.rectangle(err_img, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 0, 255), 4)
            err_img = draw_chinese_text(err_img, "模型漏抓", (int(box[0]), int(box[1])-35), color=(0, 0, 255), size=25)
        for gt_idx, pred_idx in cls_error_indices:
            box = pred_boxes[pred_idx]
            cv2.rectangle(err_img, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 255), 4)
            err_img = draw_chinese_text(err_img, "類別誤判", (int(box[0]), int(box[1])-35), color=(0, 255, 255), size=25)
        for idx in extra_indices:
            box = pred_boxes[idx]
            cv2.rectangle(err_img, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 165, 255), 4)
            err_img = draw_chinese_text(err_img, "標註漏標", (int(box[0]), int(box[1])-35), color=(0, 165, 255), size=25)

        cv2.imwrite(os.path.join(SAVE_DIR, f"{base_name}_{timestamp}_GT.jpg"), gt_img)
        cv2.imwrite(os.path.join(SAVE_DIR, f"{base_name}_{timestamp}_Pred.jpg"), pred_img_bgr)
        cv2.imwrite(os.path.join(SAVE_DIR, f"{base_name}_{timestamp}_Diag.jpg"), err_img)

        fig, axes = plt.subplots(1, 3, figsize=(25, 10))
        axes[0].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"1. GT ({base_name})", fontsize=15)
        axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(pred_img_bgr, cv2.COLOR_BGR2RGB))
        axes[1].set_title("2. YOLO Prediction", fontsize=15)
        axes[1].axis('off')
        axes[2].imshow(cv2.cvtColor(err_img, cv2.COLOR_BGR2RGB))
        axes[2].set_title("3. 診斷 (紅:漏抓, 黃:誤判, 橘:多抓)", fontsize=15, color='red')
        axes[2].axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("ℹ️ 未發現標註檔，僅顯示預測結果。")
        plt.figure(figsize=(10, 10))
        plt.imshow(cv2.cvtColor(pred_img_bgr, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

def find_latest_model_path(base_dir='runs/detect'):
    search_path = os.path.join(base_dir, 'train*')
    dirs = glob.glob(search_path)
    if not dirs: return 'best_weights/best.pt'
    latest_dir = max(dirs, key=os.path.getmtime)
    return os.path.join(latest_dir, 'weights', 'best.pt')

if __name__ == '__main__':
    model_path = find_latest_model_path()
    current_model = YOLO(model_path)

    if os.path.isdir(TEST_PATH):
        img_list = []
        for ext in ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG']:
            img_list.extend(glob.glob(os.path.join(TEST_PATH, ext)))
        
        # 使用 set 移除 Windows 下因大小寫不分產生的重複項目
        unique_img_list = sorted(list(set([os.path.abspath(p) for p in img_list])))
        
        print(f"📂 發現 {len(unique_img_list)} 張不重複圖片，開始批量診斷...")
        for img_p in unique_img_list:
            run_image_inference(img_p, current_model)
    elif os.path.isfile(TEST_PATH):
        run_image_inference(TEST_PATH, current_model)
    else:
        print("❌ 找不到路徑。")


In [ ]:
import sys
sys.argv = [sys.argv[0]]
import os, glob, cv2, time
import numpy as np
from ultralytics import YOLO
from tqdm.notebook import tqdm
from collections import defaultdict

def find_model_path_by_id(target_id="", base_dir="runs/detect"):
    """
    從輸入編號 (如 1c, 7) 自動搜尋匹配的權重檔案路徑，預設/為空時取最新 (*)
    """
    if not target_id or target_id == '*':
        search_patterns = [os.path.join(base_dir, 'train*'), os.path.join(base_dir, 'exp*')]
        dirs = []
        for pattern in search_patterns: dirs.extend(glob.glob(pattern))
        valid_dirs = [d for d in dirs if os.path.exists(os.path.join(d, 'weights', 'best.pt'))]
        if valid_dirs:
            latest_dir = max(valid_dirs, key=os.path.getmtime)
            return os.path.join(latest_dir, 'weights', 'best.pt')
        weights = glob.glob(os.path.join('best_weights', '*.pt'))
        return max(weights, key=os.path.getmtime) if weights else 'yolov11n.pt'
        
    search_patterns = [os.path.join(base_dir, 'train*'), os.path.join(base_dir, 'exp*')]
    dirs = []
    for pattern in search_patterns: dirs.extend(glob.glob(pattern))
    
    matched_dirs = []
    for d in dirs:
        folder_name = os.path.basename(d).lower()
        if target_id.lower() in folder_name and os.path.exists(os.path.join(d, 'weights', 'best.pt')):
            matched_dirs.append(d)
            
    if matched_dirs:
        best_match = max(matched_dirs, key=os.path.getmtime)
        return os.path.join(best_match, 'weights', 'best.pt')
        
    best_weights_dir = 'best_weights'
    if os.path.exists(best_weights_dir):
        weights = glob.glob(os.path.join(best_weights_dir, '*.pt'))
        matched_weights = [w for w in weights if target_id.lower() in os.path.basename(w).lower()]
        if matched_weights:
            return max(matched_weights, key=os.path.getmtime)
            
    if os.path.exists(target_id): return target_id
        
    print(f"⚠️ 找不到匹配編號 '{target_id}' 的權重檔，將自動切換為最新模型 (*)")
    return find_model_path_by_id('*', base_dir)

# ==========================================
# 🎬 影片偵測核心 (支援單部 / 批量 / 獨立 ID)
# ==========================================

# 1. 任務設定
SOURCE_LIST = [
    "測試模型用/模擬_4(無聲音).mp4",
]

# 2. 功能開關
VID_STRIDE     = 1
CONF_THRESHOLD = 0.5
ENABLE_SAVING  = True
ENABLE_SHOWING = False

def process_videos(sources, model_path):
    print(f"🎯 使用權重: {model_path}")
    
    save_dir = "偵測結果影片"
    if not os.path.exists(save_dir): os.makedirs(save_dir)
        
    normalized_path = model_path.replace("\\", "/")
    path_parts = normalized_path.split("/")
    if len(path_parts) >= 3 and path_parts[-2] == "weights":
        weight_name = f"{path_parts[-3]}_{os.path.splitext(path_parts[-1])[0]}"
    else:
        weight_name = os.path.splitext(path_parts[-1])[0]
        
    date_str = time.strftime("%Y%m%d")
    
    for video_source in sources:
        model = YOLO(model_path)
        
        is_url = video_source.startswith("http")
        video_name = "YouTube_Live.mp4" if is_url else os.path.basename(video_source)
        print(f"\n🚀 開始處理: {video_name}")
        
        if is_url:
            from cap_from_youtube import cap_from_youtube
            cap = cap_from_youtube(video_source, resolution='720p')
        else:
            if not os.path.exists(video_source): 
                print(f"❌ 找不到檔案: {video_source}"); continue
            cap = cv2.VideoCapture(video_source)

        w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        out = None
        if ENABLE_SAVING:
            save_filename = f"{weight_name}_{date_str}_{video_name}"
            save_path = os.path.join(save_dir, save_filename)
            out = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/VID_STRIDE, (w, h))
            print(f"💾 儲存路徑: {save_path}")

        pbar = tqdm(total=total_f if total_f>0 else None, desc=f"Processing {video_name[:15]}")
        frame_idx = 0
        max_id = 0
        
        while cap.isOpened():
            success, frame = cap.read()
            if not success: break
            
            frame_idx += 1
            pbar.update(1)
            if frame_idx % VID_STRIDE != 0: continue

            results = model.track(frame, persist=True, conf=CONF_THRESHOLD, iou=0.7, verbose=False)
            
            if results[0].boxes.id is not None:
                curr_max = int(results[0].boxes.id.max().item())
                if curr_max > max_id: max_id = curr_max
            
            annotated_frame = results[0].plot()
            cv2.putText(annotated_frame, f"Total Strawberry IDs: {max_id}", (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

            if ENABLE_SHOWING:
                try:
                    cv2.imshow("Strawberry Detection", annotated_frame)
                    if cv2.waitKey(1) & 0xFF == ord('q'): break
                except (cv2.error, Exception): pass
            
            if out: out.write(annotated_frame)

        cap.release()
        if out: out.release()
        pbar.close()
        print(f"✅ {video_name} 處理完畢，最終累計 ID: {max_id}")

    try:
        cv2.destroyAllWindows()
    except (cv2.error, Exception): pass

if __name__ == "__main__":
    print("==========================================")
    print("🎯 YOLO 權重選擇器")
    print("  - 可輸入編號 (如: 1c, 7) 或完整名稱 (如: exp1c_yolo11s_p2cbam)")
    print("  - 直接按 Enter 鍵：預設取最新的實驗權重 (*)")
    print("==========================================")
    
    try:
        target_id = input("請輸入權重名稱或編號 (預設為 *): ").strip()
    except Exception:
        target_id = ""
        
    model_path = find_model_path_by_id(target_id)
    process_videos(SOURCE_LIST, model_path=model_path)
